# Ternary Bonsai 2 27B: native PTQ1_0 target precompute + teacher-free head training (Tesla T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcpe500/legacy1070qi38/blob/main/qwen38-engine/tools/bonsai_mtp_train/train_bonsai_t4_student_colab.ipynb)

Companion to the live-training notebook — but this one is **T4-only**: there is no BF16 conversion (no
50.1 GiB teacher folder), no A100 session, and no layer-by-layer CPU<->GPU streaming. The engine itself
(`qwen38-engine`, built here for sm_75) loads the whole `Ternary-Bonsai-2-27B-PTQ1_0.gguf` (~5.54 GiB,
all 64 layers + the LM head resident, ~5.7 GiB VRAM) and runs Phase A directly:

| Phase | What | Cost |
|---|---|---|
| A (once) | `./build/qwen38-engine --dump-targets`: per sample a clean DeltaNet/KV state, one full prefill, the output-normed hidden of every position captured, top-64 logits + FP16 logsumexp reduced ON the GPU (only ~0.4 MB of ids/vals/lse crosses D2H per sample), hidden stored INT4 group-64, the FP16-dequantized LM head cached in VRAM — the exact math the home engine runs, so parity is by construction. Every shard logs a timing breakdown (prefill/head/quant/io ms + head GFLOPS). Plus `export_frozen_head.py`: only 3 tensors (embed, output_norm, lm_head — unrotated) -> `lm_head_embed_frozen.safetensors` in ~2-4 min, never touching the other ~850 tensors | ~5.8 GB Drive per 2000 samples (4000 samples ~11.6 GB); speed recorded by the first run |
| B (any number of times) | `train_bonsai_mtp.py --from-targets`: the same blk.64 student (mtp unroll-4 or df3 block-16), **renormalized top-K KL (mathematically >= 0)** + cosine loss, FP16 + GradScaler on pre-Ampere cards, 8-bit AdamW, warm start from your r3 sidecar | T4; ~2-6 s/step at 16K tokens/step is the estimate until the first run measures it |

Notes:
- Phase A reads `tokens.bin` + `plan.json` written by `tokenize_for_engine.py`: the corpus is tokenized with the
  engine's own BPE (the exact tokenizer the dump uses), balanced over the sources of the `prepare_corpus.py`
  shards (OpenCodeInstruct / OpenThoughts / UltraChat / Wikipedia-ID / Aya).
- On sm_75+ the dump dequantizes one weight matrix at a time into a small FP16 VRAM scratch (~170-335 MiB) and
  multiplies through `cublasGemmEx` (FP16 in, FP32 accumulate, Tensor Core path forced); the LM head sits in
  VRAM as a one-time FP16 cache (2.4 GiB, budget-guarded). sm_61 (the local GTX 1070 Ti) keeps the
  FP32/dp4a path untouched (SASS gate: zero half arithmetic) and runs the same GPU top-K reduction.
- Shards are plain `.npy` directories (the Python tool writes `.npz`) — `--from-targets` reads both. Every shard
  is written atomically and auto-resumed; `--max-gb` stops with a consistent manifest.
- Phase B warms from `procreations_r3` by default (the `INIT_MODE` dropdown in cell 4).

In [ ]:
#@title 1. Preflight (Tesla T4 15 GiB) and helpers
DRIVE_OUT = "/content/drive/MyDrive/bonsai_mtp"  #@param {type:"string"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
MIN_VRAM_GB = 14  #@param {type:"number"}
MIN_RAM_GB = 12  #@param {type:"number"}

import collections
import datetime
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

PY = sys.executable
REPO_DIR = "/content/legacy1070qi38"
TOOLS = f"{REPO_DIR}/qwen38-engine/tools/bonsai_mtp_train"
ENGINE = f"{REPO_DIR}/qwen38-engine/build/qwen38-engine"
TARGETS_DIR = f"{DRIVE_OUT}/targets"
CORPUS_DIR = f"{DRIVE_OUT}/corpus"
LOG_DIR = f"{DRIVE_OUT}/logs"


def run(cmd, name, cwd=None, env=None, ok_codes=(0,)):
    """Runs cmd (a list), streams its output here and into DRIVE_OUT/logs/<name>.log, raises on a bad exit code."""
    os.makedirs(LOG_DIR, exist_ok=True)
    log = os.path.join(LOG_DIR, f"{name}.log")
    full_env = dict(os.environ, PYTHONUNBUFFERED="1", **(env or {}))
    tail = collections.deque(maxlen=40)
    t0 = time.time()
    print(f"$ {' '.join(map(str, cmd))}\n  log: {log}", flush=True)
    with open(log, "a", encoding="utf-8") as lf:
        lf.write(f"\n===== {datetime.datetime.now().isoformat()} $ {' '.join(map(str, cmd))}\n")
        p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=full_env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace")
        for line in p.stdout:
            print(line, end="", flush=True)
            lf.write(line)
            tail.append(line)
        rc = p.wait()
        lf.write(f"===== exit code {rc} after {time.time() - t0:.0f} s\n")
    if rc not in ok_codes:
        raise RuntimeError(f"[{name}] failed with exit code {rc} after {time.time() - t0:.0f} s. Last lines:\n"
                           + "".join(tail) + f"\nFull log: {log}")
    print(f"[{name}] OK in {time.time() - t0:.0f} s", flush=True)
    return rc


try:
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
                       capture_output=True, text=True, check=True).stdout.strip().splitlines()
except (OSError, subprocess.CalledProcessError) as e:
    raise RuntimeError("No NVIDIA GPU visible. Runtime -> Change runtime type -> select a Tesla T4 (or better).") from e
gpu_name, gpu_mib, driver = [x.strip() for x in q[0].split(",")]
gpu_gb = float(gpu_mib) / 1024
print(f"GPU: {gpu_name}, {gpu_gb:.1f} GiB, driver {driver}")
if gpu_gb < MIN_VRAM_GB:
    raise RuntimeError(f"{gpu_name} has {gpu_gb:.1f} GiB; this notebook wants >= {MIN_VRAM_GB} GiB (Tesla T4 = 15 GiB).")
with open("/proc/meminfo") as f:
    mem = {l.split(":")[0]: int(l.split()[1]) for l in f}
ram_gb = mem["MemTotal"] / 1e6
print(f"RAM: {ram_gb:.1f} GB total; CPU threads {os.cpu_count()}")
if ram_gb < MIN_RAM_GB:
    raise RuntimeError(f"{ram_gb:.1f} GB RAM; the engine build + shard staging want >= {MIN_RAM_GB} GB.")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError("Google Drive did not mount at /content/drive/MyDrive.")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"DRIVE_OUT = {DRIVE_OUT} ({shutil.disk_usage(DRIVE_OUT).free / 1e9:.1f} GB free there)")
print("preflight OK")

In [ ]:
#@title 2. Clone, build qwen38-engine (sm_75), ggml/llama libs, pip, CPU self-tests
REPO_URL = "https://github.com/mcpe500/legacy1070qi38.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
LLAMA_REPO = "https://github.com/mcpe500/llama.cpp.git"  #@param {type:"string"}
LLAMA_BRANCH = "feat/iso3-kv"  #@param {type:"string"}
INSTALL_BITSANDBYTES = True  #@param {type:"boolean"}
RUN_SELFTESTS = True  #@param {type:"boolean"}

# The engine links the ggml CPU libraries of the mcpe500 fork AND the llama/llama-common libs (the CMake
# configure find_library()es them). No CUDA ggml: the engine carries its own kernels, built here for sm_75
# (CMAKE_CUDA_ARCHITECTURES=75; CUDA 13 on Colab dropped sm_61 and the CMake default no longer fights -D).
import glob as _glob
import re as _re

LLAMA_DIR = "/content/llama.cpp"
LLAMA_BUILD = f"{LLAMA_DIR}/build-iso3"
BUILD_DIR = f"{REPO_DIR}/qwen38-engine/build"
NCPU = str(os.cpu_count() or 2)

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--filter=blob:none", REPO_URL, REPO_DIR], "clone_repo")
run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], "fetch_repo")
run(["git", "-C", REPO_DIR, "checkout", BRANCH], "checkout_repo")
run(["git", "-C", REPO_DIR, "pull", "--ff-only"], "pull_repo")

if not os.path.isdir(LLAMA_DIR):
    run(["git", "clone", "--filter=blob:none", "-b", LLAMA_BRANCH, LLAMA_REPO, LLAMA_DIR], "clone_llama")
if not _glob.glob(f"{LLAMA_BUILD}/**/libllama-common.so*", recursive=True):
    run(["cmake", "-S", LLAMA_DIR, "-B", LLAMA_BUILD, "-DCMAKE_BUILD_TYPE=Release", "-DGGML_CUDA=OFF",
         "-DLLAMA_BUILD_COMMON=ON", "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_TESTS=OFF",
         "-DLLAMA_BUILD_EXAMPLES=OFF", "-DLLAMA_BUILD_SERVER=OFF"], "cmake_llama")
    run(["cmake", "--build", LLAMA_BUILD, "--target", "ggml", "llama", "llama-common", "-j", NCPU], "build_llama_all")

# Collect every built library location: -D overrides for find_library + RPATH/LD_LIBRARY_PATH so the binary
# finds libggml-cpu regardless of where the fork dropped it.
all_libs = _glob.glob(f"{LLAMA_BUILD}/**/*.so", recursive=True) + _glob.glob(f"{LLAMA_BUILD}/**/*.a", recursive=True)
lib_dirs = sorted({os.path.dirname(p) for p in all_libs})
so_dirs = sorted({os.path.dirname(p) for p in all_libs if p.endswith(".so")})
rpath = ":".join(so_dirs)
os.environ["LD_LIBRARY_PATH"] = rpath + (":" + os.environ["LD_LIBRARY_PATH"] if os.environ.get("LD_LIBRARY_PATH") else "")


def pick_lib(*patterns):
    for pat in patterns:
        hits = sorted(_glob.glob(f"{LLAMA_BUILD}/**/{pat}", recursive=True))
        if hits:
            return hits[0]
    return ""


extra_cmake = [f"-DCMAKE_LIBRARY_PATH={';'.join(lib_dirs)}", f"-DCMAKE_BUILD_RPATH={';'.join(so_dirs)}"]
for var, pats in [("GGML_LIB", ("libggml.so", "libggml.a")),
                  ("LLAMA_LIB", ("libllama.so", "libllama.a")),
                  ("LLAMA_COMMON_LIB", ("libllama-common.so", "libllama-common.a", "libcommon.so", "libcommon.a"))]:
    found = pick_lib(*pats)
    if found:
        extra_cmake.append(f"-D{var}={found}")

if os.path.isdir(BUILD_DIR):
    shutil.rmtree(BUILD_DIR)   # stale caches (an old CMAKE_CUDA_ARCHITECTURES) must not survive a runtime change

cmake_env = {"LLAMA_SRC": LLAMA_DIR, "LLAMA_BUILD": LLAMA_BUILD, "LD_LIBRARY_PATH": os.environ["LD_LIBRARY_PATH"]}
run(["cmake", "-S", f"{REPO_DIR}/qwen38-engine", "-B", BUILD_DIR, "-DCMAKE_BUILD_TYPE=Release",
     "-DCMAKE_CUDA_ARCHITECTURES=75", f"-DLLAMA_SRC={LLAMA_DIR}", f"-DLLAMA_BUILD={LLAMA_BUILD}", *extra_cmake],
    "cmake_engine", env=cmake_env)
run(["cmake", "--build", BUILD_DIR, "--target", "qwen38-engine", "-j", NCPU], "build_engine", env=cmake_env)
run([ENGINE], "engine_link_check", ok_codes=(2,))   # usage line, exit 2 = expected: links + runs

run([PY, "-m", "pip", "install", "-q", "gguf", "safetensors", "huggingface_hub"], "pip_core")
if INSTALL_BITSANDBYTES:
    try:
        run([PY, "-m", "pip", "install", "-q", "bitsandbytes"], "pip_bitsandbytes")
    except RuntimeError as e:
        print(f"WARNING: bitsandbytes unavailable, --optim adamw8bit falls back to Adafactor.\n{e}")
if RUN_SELFTESTS:
    for s in ("precompute_teacher_targets", "train_bonsai_mtp", "prepare_corpus", "convert_gguf_to_hf_bf16",
              "inject_mtp_gguf", "export_frozen_head", "tokenize_for_engine"):
        run([PY, f"{TOOLS}/{s}.py", "--selftest"], f"selftest_{s}", env={"CUDA_VISIBLE_DEVICES": ""})
print("cell 2 OK (engine built for sm_75)")

In [ ]:
#@title 3. Phase A (once): native PTQ1_0 dump on the T4 -> target package on Drive
RUN_PHASE_A = True  #@param {type:"boolean"}
HF_REPO = "prism-ml/Ternary-Bonsai-2-27B-gguf"  #@param {type:"string"}
GGUF_NAME = "Ternary-Bonsai-2-27B-PTQ1_0.gguf"  #@param {type:"string"}
SAMPLES = 4000  #@param {type:"integer"}
SEQ_LEN = 1024  #@param {type:"integer"}
TOP_K = 64  #@param {type:"integer"}
HIDDEN_DTYPE = "int4g64"  #@param ["int4g64", "int8"]
SHARD_SAMPLES = 64  #@param {type:"integer"}
MAX_GB = 40  #@param {type:"number"}
VRAM_BUDGET_MIB = 14000  #@param {type:"integer"}
DUMP_F32_HIDDEN = False  #@param {type:"boolean"}

# Native all-resident dump: ~5.7 GiB VRAM for the quantized 27B + the FP16 head cache (~2.4 GiB), NO BF16,
# NO layer streaming, top-K/lse reduced on the GPU. Auto-resume per shard; the per-shard log prints a
# timing breakdown (prefill/head/quant/io ms) and an honest ETA (remaining tokens / measured rate).
from huggingface_hub import hf_hub_download
import json as _json


def ensure_gguf():
    gguf_path = f"/content/models/{GGUF_NAME}"
    os.makedirs("/content/models", exist_ok=True)
    if not os.path.exists(gguf_path) or os.path.getsize(gguf_path) < 5_000_000_000:
        print(f"Downloading {HF_REPO}/{GGUF_NAME} -> /content/models ...", flush=True)
        hf_hub_download(repo_id=HF_REPO, filename=GGUF_NAME, local_dir="/content/models")
    print(f"GGUF ready: {gguf_path} ({os.path.getsize(gguf_path) / 1e9:.2f} GB)", flush=True)
    return gguf_path


def manifest_done(target_n):
    p = os.path.join(TARGETS_DIR, "manifest.json")
    if not os.path.exists(p):
        return False
    try:
        return _json.load(open(p)).get("total_samples", 0) >= target_n
    except (ValueError, OSError):
        return False


gguf = f"/content/models/{GGUF_NAME}"
tok_bin = "/content/tok/tokens.bin"

if not RUN_PHASE_A:
    print("RUN_PHASE_A = False: skipped (frozen-head file still exported below if missing)")
else:
    if not os.path.isdir(CORPUS_DIR) or not os.listdir(CORPUS_DIR):
        raise RuntimeError(f"no corpus at {CORPUS_DIR}: run prepare_corpus.py once (main notebook) or "
                           f"point CORPUS_DIR at your prepared corpus")
    gguf = ensure_gguf()
    if not os.path.exists(tok_bin) or os.path.getsize(tok_bin) == 0:
        run([PY, f"{TOOLS}/tokenize_for_engine.py", "--corpus", CORPUS_DIR, "--gguf", gguf, "--engine", ENGINE,
             "--out", "/content/tok", "--samples", str(SAMPLES), "--seq-len", str(SEQ_LEN)], "tokenize")
    else:
        print(f"Using existing {tok_bin} ({os.path.getsize(tok_bin) // (4 * SEQ_LEN)} samples x {SEQ_LEN} tok)")

    avail_samples = os.path.getsize(tok_bin) // (4 * SEQ_LEN)
    dump_samples = min(SAMPLES, avail_samples) if avail_samples > 0 else SAMPLES

    if manifest_done(dump_samples):
        print(f"manifest at {TARGETS_DIR} already covers {dump_samples} samples - Phase A skipped")
    else:
        cmd = [ENGINE, "-m", gguf, "--dump-targets", TARGETS_DIR, "--tokens-bin", tok_bin,
               "--dump-samples", str(dump_samples), "--dump-seq-len", str(SEQ_LEN), "--dump-top-k", str(TOP_K),
               "--dump-hidden-dtype", HIDDEN_DTYPE, "--shard-samples", str(SHARD_SAMPLES),
               "--max-gb", str(MAX_GB), "--vram-budget-mib", str(VRAM_BUDGET_MIB)]
        if DUMP_F32_HIDDEN:
            cmd.append("--dump-f32-hidden")
        run(cmd, "dump_targets")

frozen = os.path.join(TARGETS_DIR, "lm_head_embed_frozen.safetensors")
if not os.path.exists(frozen):
    run([PY, f"{TOOLS}/export_frozen_head.py", "--gguf", gguf, "--out", frozen],
        "export_frozen_head", env={"CUDA_VISIBLE_DEVICES": ""})
man_p = os.path.join(TARGETS_DIR, "manifest.json")
if os.path.exists(man_p):
    man = _json.load(open(man_p))
    print(f"package: {man['total_tokens'] / 1e6:.2f}M tokens, {man['total_bytes'] / 1e9:.1f} GB, "
          f"top_k {man.get('top_k')}, hidden {man.get('hidden_dtype')}")
else:
    print(f"no manifest at {TARGETS_DIR} yet - Phase B needs it (run Phase A first)")

In [ ]:
#@title 4. Phase B: teacher-free student training from the package (T4, FP16 + GradScaler)
RUN_PHASE_B = True  #@param {type:"boolean"}
DRAFTER = "mtp"  #@param ["mtp", "df3"]
INIT_MODE = "procreations_r3"  #@param ["procreations_r3", "drive_safetensors", "qwen_blk64", "random"]
INIT_HF_REPO = "ProCreations/Ternary-Bonsai-2-27B-MTP"  #@param {type:"string"}
INIT_HF_FILE = "model_mtp.safetensors"  #@param {type:"string"}
INIT_DRIVE_PATH = "/content/drive/MyDrive/bonsai_mtp/final/mtp_blk64.safetensors"  #@param {type:"string"}
STEPS = 2000  #@param {type:"integer"}
BATCH = 2  #@param {type:"integer"}
GRAD_ACCUM = 8  #@param {type:"integer"}
LR = 2e-4  #@param {type:"number"}
WARMUP = 100  #@param {type:"integer"}
EPOCHS = 3  #@param {type:"integer"}
OPTIM = "adamw8bit"  #@param ["adamw8bit", "adafactor", "adamw"]
UNROLL = 4  #@param {type:"integer"}
DF3_BLOCK = 16  #@param {type:"integer"}
SAVE_EVERY = 25  #@param {type:"integer"}
SAVE_EVERY_MIN = 20  #@param {type:"number"}
STAGE_DIR = "/content/target_cache"  #@param {type:"string"}
RUN_DIR = "/content/drive/MyDrive/bonsai_mtp/run_t4"  #@param {type:"string"}
RESUME_FROM = "auto"  #@param ["auto", "none"]

# TOP_K / HIDDEN_DTYPE come from the package manifest (never from a skipped cell), so this cell is safe to
# run on its own after Phase A produced the package once.
import json as _json
from huggingface_hub import hf_hub_download

man_p = os.path.join(TARGETS_DIR, "manifest.json")
if not os.path.exists(man_p):
    raise RuntimeError(f"no target package at {TARGETS_DIR}: run cell 3 (Phase A) first - it writes the manifest")
man = _json.load(open(man_p))
TOP_K = int(man.get("top_k", 64))
HIDDEN_DTYPE = str(man.get("hidden_dtype", "int4g64"))
print(f"package: {man['total_tokens'] / 1e6:.2f}M tokens, {man['total_bytes'] / 1e9:.1f} GB; "
      f"loss width top_k={TOP_K}, hidden {HIDDEN_DTYPE}")
if not RUN_PHASE_B:
    print("RUN_PHASE_B = False: skipped")
else:
    init_args = []
    if INIT_MODE == "procreations_r3":
        init_st = f"/content/models/{INIT_HF_FILE}"
        if not os.path.exists(init_st):
            os.makedirs("/content/models", exist_ok=True)
            print(f"Downloading {INIT_HF_REPO}/{INIT_HF_FILE} -> /content/models ...", flush=True)
            hf_hub_download(repo_id=INIT_HF_REPO, filename=INIT_HF_FILE, local_dir="/content/models")
        init_args = ["--init-mtp-safetensors", init_st]
    elif INIT_MODE == "drive_safetensors":
        if not os.path.exists(INIT_DRIVE_PATH):
            raise RuntimeError(f"--init safetensors not found: {INIT_DRIVE_PATH}")
        init_args = ["--init-mtp-safetensors", INIT_DRIVE_PATH]
    elif INIT_MODE == "qwen_blk64":
        blk64 = "/content/drive/MyDrive/bonsai_mtp/init/mtp_init_qwen_blk64.safetensors"
        if not os.path.exists(blk64):
            raise RuntimeError(f"{blk64} missing: export blk.64 of the qwen GGUF once with "
                               f"train_bonsai_mtp.py --export-init and put it there")
        init_args = ["--init-mtp-safetensors", blk64]
    run([PY, f"{TOOLS}/train_bonsai_mtp.py", "--from-targets", TARGETS_DIR,
         "--lm-head-file", os.path.join(TARGETS_DIR, "lm_head_embed_frozen.safetensors"),
         "--out", RUN_DIR, "--drafter", str(DRAFTER), "--unroll", str(UNROLL), "--df3-block", str(DF3_BLOCK),
         "--steps", str(STEPS), "--seq-len", "1024", "--batch", str(BATCH), "--grad-accum", str(GRAD_ACCUM),
         "--lr", str(LR), "--warmup", str(WARMUP), "--epochs", str(EPOCHS), "--optim", str(OPTIM),
         "--top-k-kl", str(TOP_K), "--hidden-dtype", HIDDEN_DTYPE, "--save-every", str(SAVE_EVERY),
         "--save-every-min", str(SAVE_EVERY_MIN), "--keep-last", "2", "--resume-from", str(RESUME_FROM),
         "--log-every", "10", "--device", "cuda", "--stage-dir", STAGE_DIR] + init_args, "train_t4")
    with open(os.path.join(RUN_DIR, "train_log.jsonl")) as f:
        last = _json.loads(f.readlines()[-1])
    print(f"last step {last['step']}: top1_agree {[round(x, 3) for x in last['top1_agree']]} "
          f"({last.get('sec_per_step', '?')} s/step, eta {last.get('eta_h', '?')} h)")

In [ ]:
#@title 5. Export the trained head to sidecar GGUFs (same plug-and-play file as always)
EXPORT_QUANTS = "q4_k_m,q8_0"  #@param {type:"string"}
FINAL = f"{DRIVE_OUT}/final_t4"

latest = subprocess.run([PY, f"{TOOLS}/train_bonsai_mtp.py", "--latest", RUN_DIR], capture_output=True, text=True)
if latest.returncode != 0:
    raise RuntimeError(f"no checkpoint in {RUN_DIR}: run cell 4 first.\n{latest.stderr}")
ckpt = latest.stdout.strip()
print(f"checkpoint: {ckpt}")
os.makedirs(FINAL, exist_ok=True)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()


lines = []
for qt in [q.strip() for q in EXPORT_QUANTS.split(",") if q.strip()]:
    out = os.path.join(FINAL, f"mtp_blk64_{qt}.gguf")
    run([PY, f"{TOOLS}/inject_mtp_gguf.py", "--from-checkpoint", ckpt, "--quant", qt, "--mtp-only-out", out],
        f"export_{qt}", env={"CUDA_VISIBLE_DEVICES": ""})
    lines.append(f"mtp_blk64_{qt}.gguf  {os.path.getsize(out)} B  sha256 {sha256(out)}")
qt0 = [q.strip() for q in EXPORT_QUANTS.split(",") if q.strip()][0]
print("\n".join(lines))
print(f"""
Sidecar GGFs on Drive: {FINAL}. On your PC (engine built with ./scripts/setup.sh):

  taskset -c 0,1,2,3 ./build/qwen38-engine -m ~/models/27b-bonsai/Ternary-Bonsai-2-27B-PTQ1_0.gguf \\
      --mtp-gguf ~/models/27b-bonsai/mtp_blk64_{qt0}.gguf --gpu-layers auto --ctx 4096 \\
      --mtp-adaptive 1 --gen-mtp-steps 2 -f bench/prompt512.txt -n 256
""")

## Troubleshooting (T4 notebook)

- **Cell 2 build fails:** the engine needs the ggml CPU libraries of the `mcpe500/llama.cpp` fork
  (`feat/iso3-kv`); the cell clones and builds them automatically (~3-8 min on the T4's 2 vCPUs, once per
  session). The engine build itself targets `CMAKE_CUDA_ARCHITECTURES=75`; `LLAMA_SRC`/`LLAMA_BUILD` are passed
  through the environment. A "GGML_LIB not found" means the llama.cpp build failed — open its log in Drive.
- **Phase A says "no corpus":** Phase A tokenizes from the `prepare_corpus.py` shards on Drive (balanced over
  OpenCodeInstruct / OpenThoughts / UltraChat / Wikipedia-ID / Aya). Run the main notebook's corpus step once;
  it lands in `DRIVE_OUT/corpus`.
- **HF download fails:** set `HF_REPO` / `INIT_HF_REPO` to the repos that hold your
  `Ternary-Bonsai-2-27B-PTQ1_0.gguf` (5,946,648,928 bytes) and your warm-start `model_mtp.safetensors`.
- **CUDA out of memory in cell 3:** keep `VRAM_BUDGET_MIB >= 14000` (the resident 27B is ~5.7 GiB plus the FP16
  scratch); lower `SEQ_LEN` (1024 -> 512) if the runtime is smaller than a full T4.
- **Slow steps in cell 4 with Drive shards:** keep `STAGE_DIR` at local SSD (`/content/target_cache`), not Drive.
- **`bitsandbytes` WARNING in cell 2:** `--optim adamw8bit` falls back to Adafactor automatically.
- **KL is >= 0:** the sparse top-K loss is the renormalized teacher distribution
  (`softmax(topk_vals)` vs the student's renormalized K logits) — a true KL divergence. `top1_agree` is the
  number to watch (the engine's acceptance proxy), then the real engine race.